In [4]:
import os
import numpy as np
import torch
import xml.etree.ElementTree as ET
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch # Đảm bảo đã import torch để kiểm tra kiểu dữ liệu
import cv2
import os
import cv2
import torch
import numpy as np
import xml.etree.ElementTree as ET
from collections import Counter
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.utils.tensorboard import SummaryWriter
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_320_fpn, FasterRCNN_MobileNet_V3_Large_320_FPN_Weights
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm.autonotebook import tqdm
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torch.utils.data import Subset



In [5]:
import os
import cv2
import torch
import xml.etree.ElementTree as ET
from torch.utils.data import Dataset
import numpy as np

class ISICDataset(Dataset):
    def __init__(self, root, subset, transforms=None):
        self.root = os.path.join(root, subset)
        self.transforms = transforms
        
        # 1. Lấy danh sách ảnh và sắp xếp để đảm bảo tính nhất quán
        self.imgs = sorted([f for f in os.listdir(self.root) 
                           if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        
        # 2. Tự động tạo Label Map từ dữ liệu thực tế
        self.label_map = self._create_label_map()
        
    def _create_label_map(self):
        unique_labels = set()
        xml_files = [f for f in os.listdir(self.root) if f.lower().endswith('.xml')]
        for xml_file in xml_files:
            tree = ET.parse(os.path.join(self.root, xml_file))
            root = tree.getroot()
            for obj in root.findall('object'):
                unique_labels.add(obj.find('name').text)
        
        # ID 0 dành cho background, các bệnh từ 1-9
        mapping = {name: i + 1 for i, name in enumerate(sorted(list(unique_labels)))}
        mapping["background"] = 0
        return mapping

    def __getitem__(self, idx):
        # Đường dẫn ảnh và XML tương ứng
        img_path = os.path.join(self.root, self.imgs[idx])
        xml_path = os.path.join(self.root, os.path.splitext(self.imgs[idx])[0] + ".xml")

        # Đọc ảnh bằng OpenCV và chuyển sang RGB
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        tree = ET.parse(xml_path)
        root = tree.getroot()
        
        boxes = []
        labels = []
        for obj in root.findall('object'):
            name = obj.find('name').text
            labels.append(self.label_map[name])
            
            bbox = obj.find('bndbox')
            xmin = float(bbox.find('xmin').text)
            ymin = float(bbox.find('ymin').text)
            xmax = float(bbox.find('xmax').text)
            ymax = float(bbox.find('ymax').text)
            
            # Kiểm tra tính hợp lệ của box để tránh lỗi tọa độ âm hoặc bằng không
            if xmax > xmin and ymax > ymin:
                boxes.append([xmin, ymin, xmax, ymax])

        # Áp dụng Albumentations (nếu có)
        if self.transforms:
            transformed = self.transforms(image=img, bboxes=boxes, labels=labels)
            img = transformed['image']
            boxes = transformed['bboxes']
            labels = transformed['labels']

        # --- PHẦN SỬA LỖI QUAN TRỌNG CHO FASTER R-CNN ---
        
        # Đảm bảo boxes luôn có dạng [N, 4], kể cả khi rỗng
        boxes = torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        
        # Tính toán diện tích (area) trực tiếp từ tensor boxes
        if boxes.shape[0] > 0:
            area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
        else:
            area = torch.tensor([0], dtype=torch.float32)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
            "area": area,
            "iscrowd": torch.zeros((len(labels),), dtype=torch.int64)
        }

        # Chuyển đổi ảnh sang tensor float32 và chuẩn hóa về [0, 1]
        if not isinstance(img, torch.Tensor):
            img = torch.from_numpy(img).permute(2, 0, 1).to(torch.float32) / 255.0
        elif img.dtype != torch.float32:
            img = img.to(torch.float32) / 255.0

        return img, target

    def __len__(self):
        return len(self.imgs)

# Hàm ghép nối Batch tùy chỉnh cho Object Detection
def collate_fn(batch):
    return tuple(zip(*batch))

In [6]:
def collate_fn(batch):
    return tuple(zip(*batch))

def get_primary_label(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    obj = root.find('object')
    return obj.find('name').text if obj is not None else "__empty__"


# ... (Giữ nguyên phần import và class ISICDataset)

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print (device)
    BASE_DIR = r"D:\xu_li_anh\btl\data"
    OUTPUT_DIR = r"D:\xu_li_anh\btl\checkpoin2"
    LOG_DIR = os.path.join(OUTPUT_DIR, "logs")
    MODEL_DIR = os.path.join(OUTPUT_DIR, "models")
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ================= AUGMENTATION (Giữ nguyên) =================
    train_transform = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Rotate(limit=30, p=0.5),
        A.OneOf([A.GaussianBlur(p=0.3), A.MotionBlur(p=0.3)], p=0.3),
        A.OneOf([A.RandomBrightnessContrast(p=0.5), A.RandomGamma(p=0.5)], p=0.5),
        A.HueSaturationValue(p=0.3),
        ToTensorV2()
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

    val_transform = A.Compose([ToTensorV2()], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

    # ================= DATASET & SAMPLER FIX =================
    train_ds = ISICDataset(root=BASE_DIR, subset='train', transforms=train_transform)
    valid_ds = ISICDataset(root=BASE_DIR, subset='valid', transforms=val_transform)

    label_map = train_ds.label_map
    num_classes = len(label_map)
    print(f"Detected Classes: {label_map}")

    print("--- Đang thống kê nhãn để cân bằng dữ liệu tập Train ---")
    train_xml_paths = [os.path.join(train_ds.root, os.path.splitext(f)[0] + ".xml") for f in train_ds.imgs]
    primary_labels = [get_primary_label(p) for p in train_xml_paths]

    # FIX: Giữ lại một ít mẫu rỗng để tránh False Positive (Nhận nhầm da lành thành bệnh)
    # Ở đây tôi lọc indices để vẫn lấy ảnh, nhưng đảm bảo sampler nhận diện đúng
    filtered = [(lbl, i) for i, lbl in enumerate(primary_labels)] 
    labels_filtered = [x[0] for x in filtered]
    indices_filtered = [x[1] for x in filtered]

    class_counts = Counter(labels_filtered)
    # FIX: Tránh chia cho 0 nếu có class trống
    class_weights = {cls: 1.0 / count for cls, count in class_counts.items() if count > 0}
    sample_weights = [class_weights.get(lbl, 0) for lbl in labels_filtered]

    train_subset = Subset(train_ds, indices_filtered)

    train_sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights)*2,
        replacement=True
    )

    train_loader = DataLoader(train_subset, batch_size=4, sampler=train_sampler, collate_fn=collate_fn, num_workers=0)
    valid_loader = DataLoader(valid_ds, batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=0)

    # ================= MODEL =================
    model = fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_channels = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_channels, num_classes)

    for param in model.backbone.parameters():
        param.requires_grad = False

    model.to(device)

    # ================= TRAIN CONFIG =================
    optimizer = torch.optim.AdamW([
            {"params": model.backbone.parameters(), "lr": 5e-6},
            {"params": model.roi_heads.parameters(), "lr": 1e-4},
        ], weight_decay=0.0005)

    lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
    writer = SummaryWriter(log_dir=LOG_DIR)
    map_metric = MeanAveragePrecision(iou_type="bbox")

    NUM_EPOCHS = 50
    best_map = 0.0
    patience = 5
    no_improve = 0

    for epoch in range(NUM_EPOCHS):
        # FIX: Sửa lỗi thụt đầu dòng (Indentation) và Logic Reset Optimizer
        if epoch == 5:
            print(">>> Unfreezing backbone...")
            for param in model.backbone.parameters():
                param.requires_grad = True
            
            # Cấu hình lại optimizer một lần duy nhất khi mở khóa
            optimizer = torch.optim.AdamW([
                {"params": model.backbone.parameters(), "lr": 3e-6},
                {"params": model.roi_heads.parameters(), "lr": 8e-5},
            ], weight_decay=0.0005)
            
            lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS - epoch)

        # ================= TRAINING PHASE (Giữ nguyên) =================
        model.train()
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]", colour="green")
        epoch_losses = []

        for images, targets in train_pbar:
            images = [img.to(device, dtype=torch.float32) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

            optimizer.zero_grad()
            losses.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0) 
            optimizer.step()

            epoch_losses.append(losses.item())
            train_pbar.set_postfix({"loss": f"{np.mean(epoch_losses):.4f}"})

        avg_loss = np.mean(epoch_losses)
        writer.add_scalar("Loss/Train", avg_loss, epoch)
        lr_scheduler.step()

        # ================= EVALUATION PHASE =================
        model.eval()
        map_metric.reset()

        with torch.no_grad():
            for images, targets in valid_loader:
                images = [img.to(device, dtype=torch.float32) for img in images]
                outputs = model(images)

                # FIX: Chỉ lọc threshold khi đánh giá để mAP chuẩn xác hơn theo COCO
                preds = []
                for o in outputs:
                    # Giữ nguyên logic lọc của bạn nhưng khuyến khích truyền hết vào map_metric
                    keep = o["scores"] > 0.05 # Để thấp hơn 0.3 để tính mAP chính xác
                    preds.append({
                        "boxes": o["boxes"][keep].cpu(),
                        "scores": o["scores"][keep].cpu(),
                        "labels": o["labels"][keep].cpu()
                    })

                gts = [{"boxes": t["boxes"].cpu(), "labels": t["labels"].cpu()} for t in targets]
                map_metric.update(preds, gts)

        results = map_metric.compute()
        current_map = results['map'].item()
        map50 = results['map_50'].item()
        
        print(f"Epoch {epoch+1}: Loss={avg_loss:.4f} | mAP={current_map:.4f} | mAP50={map50:.4f}")

        # ================= SAVE & EARLY STOP (Giữ nguyên) =================
        if current_map > best_map:
            best_map = current_map
            no_improve = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'map': best_map,
                'label_map': label_map
            }, os.path.join(MODEL_DIR, "best_model.pth"))
            print(f"Saved best model: {best_map:.4f}")
        else:
            no_improve += 1

        if no_improve >= patience:
            print("Early stopping triggered")
            break

    writer.close()
    print("Huấn luyện hoàn tất!")

cuda


d:\xu_li_anh\venv\lib\site-packages\albumentations\core\composition.py:331: UserWarning: Got processor for bboxes, but no transform to process it.
  self._set_keys()


Detected Classes: {'actinic keratosis': 1, 'basal cell carcinoma': 2, 'dermatofibroma': 3, 'melanoma': 4, 'nevus': 5, 'pigmented benign keratosis': 6, 'seborrheic keratosis': 7, 'squamous cell carcinoma': 8, 'vascular lesion': 9, 'background': 0}
--- Đang thống kê nhãn để cân bằng dữ liệu tập Train ---


Epoch 1/50 [Train]: 100%|██████████| 357/357 [03:35<00:00,  1.66it/s, loss=0.2287]


Epoch 1: Loss=0.2287 | mAP=0.1097 | mAP50=0.2506
Saved best model: 0.1097


...

Epoch 4: Loss=0.1719 | mAP=0.1487 | mAP50=0.3486
Saved best model: 0.1487


Epoch 5/50 [Train]: 100%|██████████| 357/357 [03:13<00:00,  1.84it/s, loss=0.1714]


Epoch 5: Loss=0.1714 | mAP=0.1765 | mAP50=0.3764
Saved best model: 0.1765
>>> Unfreezing backbone...


Epoch 6/50 [Train]:   6%|▌         | 20/357 [07:22<2:04:15, 22.12s/it, loss=0.1631]


KeyboardInterrupt: 